# 🎬 AI 유튜브 롱폼 공장 — 핸드폰/코랩 버전
PC 없이 **구글 서버에서 무료로** 영상을 만들어요. 핸드폰 브라우저(크롬)로 열어서 쓰면 됩니다.

**쓰는 법**: 위에서부터 각 칸 왼쪽의 ▶ 버튼을 차례로 누르세요.
1. **① 준비** (처음 한 번) → 구글 드라이브 연결 허용
2. **② 설정** → Gemini 무료 키 입력 ([여기서 발급](https://aistudio.google.com/apikey))
3. **③ 간단 모드**로 바로 영상 만들기 **또는** **④ 화면 모드**로 PC와 똑같은 화면 쓰기

⚠️ 만드는 동안 **화면을 끄거나 다른 앱으로 오래 나가면** 무료 코랩이 끊길 수 있어요. 결과물은 구글 드라이브 `내 드라이브/유튜브공장` 폴더에 저장됩니다.


In [ ]:
#@title ① 준비하기 (처음 한 번, 1~2분 걸려요) { display-mode: "form" }
드라이브에_저장 = True #@param {type:"boolean"}
import os, sys
if not os.path.exists('/content/app'):
    !git clone -q --depth 1 -b main https://github.com/wow7003-dot/qhdwk1004-.git /content/app
else:
    !cd /content/app && git pull -q
%cd /content/app/ytfactory
!pip install -q -r requirements.txt
if 드라이브에_저장:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/유튜브공장'
else:
    OUT = '/content/output'
os.makedirs(OUT, exist_ok=True)
os.environ['YTF_PROJECTS_DIR'] = OUT
os.environ['YTF_CONFIG'] = OUT + '/config.json'
sys.path.insert(0, '/content/app/ytfactory')
print('✅ 준비 완료! 결과 저장 위치:', OUT)


In [ ]:
#@title ② 설정 (한 번 저장하면 다음에도 유지돼요) { display-mode: "form" }
Gemini_API_키 = "" #@param {type:"string"}
이미지_소스 = "pollinations" #@param ["pollinations", "huggingface", "pexels", "pixabay", "placeholder"]
Pollinations_키_선택 = "" #@param {type:"string"}
목소리 = "ko-KR-InJoonNeural" #@param ["ko-KR-InJoonNeural", "ko-KR-HyunsuMultilingualNeural", "ko-KR-SunHiNeural"]
말_빠르기 = 0 #@param {type:"slider", min:-30, max:50, step:5}
화면_비율 = "16:9" #@param ["16:9", "9:16"]
줌_이동_효과 = True #@param {type:"boolean"}
#@markdown 줌/이동 효과를 끄면 렌더링이 3~4배 빨라져요 (무료 코랩은 CPU 가 느림).
from core.config import load_settings, save_settings
s = load_settings()
if Gemini_API_키: s.gemini_api_key = Gemini_API_키.strip()
if Pollinations_키_선택: s.pollinations_key = Pollinations_키_선택.strip()
s.llm_provider = 'gemini' if s.gemini_api_key else 'none'
s.image_provider, s.voice, s.rate = 이미지_소스, 목소리, f"{말_빠르기:+d}%"
s.aspect, s.ken_burns = 화면_비율, 줌_이동_효과
save_settings(s)
print('✅ 저장됨 | 대본 AI:', '켜짐' if s.gemini_api_key else '꺼짐 (대본을 직접 넣어야 해요)')


In [ ]:
#@title ③ 간단 모드: 주제(또는 대본 파일)만 넣고 영상 만들기 { display-mode: "form" }
주제 = "조선 최강의 이름 없는 검객 이야기" #@param {type:"string"}
목표_길이_분 = 10 #@param {type:"slider", min:1, max:30, step:1}
분위기 = "몰입감 있는 스토리텔링, 긴장감" #@param {type:"string"}
그림_스타일 = "korean martial arts anime style, dark fantasy, dramatic cinematic lighting, detailed digital painting" #@param ["korean martial arts anime style, dark fantasy, dramatic cinematic lighting, detailed digital painting", "photorealistic cinematic film still, 35mm, moody lighting, ultra detailed", "joseon dynasty korea historical drama, cinematic, realistic painting, warm candle light", "soft watercolor storybook illustration, pastel colors", "3d animated movie style, vibrant colors, soft lighting", "dark horror atmosphere, foggy, eerie, cinematic, film grain", "clean modern editorial illustration, flat design, minimal"] {allow-input: true}
#@markdown **내 대본을 쓰려면**: 드라이브 `유튜브공장` 폴더에 txt 파일을 올리고 이름을 적으세요 (비워두면 AI 가 대본 작성)
대본_파일 = "" #@param {type:"string"}
#@markdown **배경음악**: 드라이브 `유튜브공장` 폴더에 mp3 를 올리고 이름을 적으세요 (선택)
BGM_파일 = "" #@param {type:"string"}
핸드폰에_바로_다운로드 = False #@param {type:"boolean"}

from pathlib import Path
from IPython.display import Image, display
from core.config import load_settings
from core import pipeline
s = load_settings()
script = ''
if 대본_파일:
    p = Path(대본_파일) if 대본_파일.startswith('/') else Path(OUT) / 대본_파일
    script = p.read_text(encoding='utf-8')
bgm = None
if BGM_파일:
    bgm = Path(BGM_파일) if BGM_파일.startswith('/') else Path(OUT) / BGM_파일
pj = pipeline.plan(s, topic=주제, script_text=script, minutes=목표_길이_분, tone=분위기, style=그림_스타일)
print('\n📜 제목:', pj['meta'].get('title'), '| 장면', len(pj['scenes']), '개\n')
pj = pipeline.render(s, pj, bgm=bgm)
display(Image(pj['thumbnail'], width=480))
print(open(Path(pj['dir']) / 'upload_info.txt', encoding='utf-8').read())
print('🎉 완성! 드라이브 저장 위치:', pj['dir'])
if 핸드폰에_바로_다운로드:
    from google.colab import files
    files.download(pj['video']); files.download(pj['thumbnail'])


In [ ]:
#@title ④ 화면 모드: PC와 똑같은 프로그램 화면을 핸드폰에서 열기 { display-mode: "form" }
#@markdown ▶ 누르고 30초쯤 기다리면 **https://….trycloudflare.com** 주소가 나와요. 눌러서 열면 됩니다.<br>
#@markdown 이 칸이 실행되는 동안만 접속돼요 (멈추면 화면도 꺼짐). 주소를 아는 사람은 누구나 접속할 수 있으니 공유하지 마세요.
import subprocess, time, re, os
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
subprocess.run('pkill -f "streamlit run"; pkill -f cloudflared', shell=True)
st_proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true', '--server.port', '8501',
                            '--server.maxUploadSize', '300'], cwd='/content/app/ytfactory',
                           stdout=open('/content/st.log', 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(5)
cf = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in cf.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('👉 이 주소를 누르세요:', url)
print('(이 칸을 멈추면 접속이 끊겨요)')
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    cf.terminate(); st_proc.terminate(); print('종료했습니다.')
